# 03 · Prompt evaluation

**Goal:** stop guessing whether a prompt is good. Build a small pipeline that gives a prompt a score, then use it to compare three versions.

| | |
|---|---|
| **You will learn** | The eval workflow, generating a test dataset, model graders, code graders, combining scores, and how far to trust the result |
| **Prerequisites** | [Notebook 01](01_conversations_and_control.ipynb) and [notebook 02](02_structured_outputs.ipynb) |
| **Run time** | About 3 minutes |
| **Cost** | About $0.11 when I ran it. This notebook stops itself at $0.30 |
| **Paid cells** | Sections 1, 2, 3, 5, 6 and 7. Sections 4 and 8 are free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## Why bother?

After writing a prompt, you usually do one of three things:

| Option | What you do | The risk |
|---|---|---|
| 1 | Try it once, decide it's fine | It breaks on the first input you didn't think of |
| 2 | Try it a few times, fix a corner case or two | Real users still surprise you |
| **3** | **Score it automatically on many inputs, then improve it** | More work up front, but you get evidence |

An eval is an exam for your prompt: a fixed set of questions (the **dataset**) and a marking scheme (the **graders**). Change the prompt, sit the same exam again, and compare the scores.

```
 1. Draft a prompt ──▶ 2. Build a dataset ──▶ 3. Run the prompt on every case
                                                         │
 5. Change the prompt and repeat ◀── 4. Grade the outputs ◀┘
```

**The prompt we'll test:** a small assistant that helps with AWS tasks. It must return **only** Python, JSON or a regex, with no explanation around it, because another program will use the output.

## Setup

The same setup as the earlier notebooks. Run the cells from top to bottom.

In [1]:
import ast
import json
import os
import re
from pathlib import Path
from statistics import mean
from typing import Literal

from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [3]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.30     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""

## 1. The dataset (paid, once)

A dataset is a list of inputs your prompt will face. You can write one by hand or have Claude write it. We use structured outputs from notebook 02, so it comes back as clean data.

Each test case has a `task`, and a `format` saying which kind of answer it needs. The code grader uses `format` later on.

**The dataset is saved to a file** and reused from then on. That matters for two reasons: you only pay for it once, and every prompt version sits **the same exam**, so the scores are comparable. To generate a new one, delete `eval_dataset.json`.

> The course suggests a cheaper model for this step. Claude Haiku would work well here if you want to save money, but this repo stays on Sonnet 5.5 throughout.

In [4]:
class TestCase(BaseModel):
    task: str
    format: Literal["python", "json", "regex"]


class Dataset(BaseModel):
    cases: list[TestCase]


def generate_dataset():
    prompt = """
Generate an evaluation dataset for prompts that produce Python, JSON or a regex for AWS-related tasks.

Generate 4 test cases, with at least one of each format.
* Each task must be solvable with a single Python function, a single JSON object, or a single regex
* Keep the tasks small: none should need much code
"""
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
        output_format=Dataset,
    ))
    return [case.model_dump() for case in response.parsed_output.cases]


DATASET_PATH = Path("eval_dataset.json")

if DATASET_PATH.exists():
    dataset = json.loads(DATASET_PATH.read_text())
    print(f"Loaded the saved dataset, which costs nothing: {DATASET_PATH}\n")
else:
    dataset = generate_dataset()
    DATASET_PATH.write_text(json.dumps(dataset, indent=2))
    print(f"Generated and saved a new dataset: {DATASET_PATH}\n")

for case in dataset:
    print(f"[{case['format']:<6}] {case['task']}")

assert len(dataset) >= 3
assert {case["format"] for case in dataset} == {"python", "json", "regex"}

[input=510 output=295 cost=$0.00397 | spent $0.0040 of $0.30]
Generated and saved a new dataset: eval_dataset.json

[python] Write a Python function `list_bucket_keys(bucket_name)` that uses boto3 to return a list of all object keys in the given S3 bucket, handling pagination.
[json  ] Create an IAM policy document as a JSON object that allows read-only access (s3:GetObject and s3:ListBucket) to a bucket named 'my-data-bucket' and all objects inside it.
[regex ] Write a regex that matches a valid AWS S3 bucket name: 3 to 63 characters, lowercase letters, numbers, dots, and hyphens, starting and ending with a letter or number.
[python] Write a Python function `parse_arn(arn)` that takes an AWS ARN string and returns a dict with the keys partition, service, region, account_id, and resource.


## 2. Running the prompt on every case (paid)

Version 1 of our prompt is deliberately plain. It says nothing about the output format.

`get_outputs` takes a dataset and a prompt function, and returns every output. Taking the prompt function as an argument is what lets us swap in other versions later without changing anything else.

In [5]:
def run_prompt_v1(test_case):
    prompt = f"Please solve the following task:\n\n{test_case['task']}"
    response = track(client.messages.create(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
    ))
    return text_from_message(response)


def get_outputs(dataset, run_prompt):
    return [{"test_case": test_case, "output": run_prompt(test_case)} for test_case in dataset]


v1_outputs = get_outputs(dataset, run_prompt_v1)

print("\nFirst output, up to 500 characters:\n")
print(v1_outputs[0]["output"][:500])

assert len(v1_outputs) == len(dataset)

[input=72 output=708 cost=$0.00722 | spent $0.0112 of $0.30]
[input=81 output=485 cost=$0.00501 | spent $0.0162 of $0.30]
[input=81 output=662 cost=$0.00678 | spent $0.0230 of $0.30]
[input=70 output=720 cost=$0.00734 | spent $0.0303 of $0.30]

First output, up to 500 characters:

Here's a function that uses boto3's built-in paginator to list every key in an S3 bucket:

```python
import boto3


def list_bucket_keys(bucket_name):
    """Return a list of all object keys in the given S3 bucket.

    Uses the list_objects_v2 paginator so buckets with more than
    1000 objects are handled correctly.
    """
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")

    keys = []
    for page in paginator.paginate(Bucket=bucket_name):
        # 'Contents'


**What to notice**

- With no format instructions, Claude likely adds code fences and an explanation. That's normally helpful, but it fails our requirement, because a program will use this output.
- We keep these outputs in `v1_outputs`, so we can grade them without paying to generate them again.

## 3. Grader 1: a model grader (paid)

A grader turns an output into a score. There are three kinds:

| Grader | How it works | Good for | Weakness |
|---|---|---|---|
| **Code** | Plain Python checks | Syntax, length, required words | Can't judge meaning |
| **Model** | Another Claude call scores the output | Quality, following instructions, helpfulness | Costs money, and scores vary |
| **Human** | A person reviews it | Anything | Slow and tedious |

First decide **what good means**. Here we have three criteria, each suited to a different grader:

| Criterion | Question | Grader |
|---|---|---|
| **Task following** | Does it do what was asked, accurately? | Model |
| **Valid syntax** | Does it parse as Python, JSON or a regex? | Code |
| **Format** | Is it only the code, with nothing around it? | Code |

**The key trick from the course:** ask for strengths, weaknesses and reasoning, as well as the score. Asked for only a number, models tend to settle on a middling score of around 6.

> **Different from the course:** the course gets clean JSON from the grader with prefill, which current models reject. We use `messages.parse()` from notebook 02. `Field(ge=1, le=10)` makes the SDK reject any score outside that range.

In [6]:
class Grade(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: int = Field(ge=1, le=10)


def grade_by_model(test_case, output):
    prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

<task>
{test_case['task']}
</task>

<solution>
{output}
</solution>

Give 1-3 key strengths, 1-3 key weaknesses, a concise reasoning, and a score from 1 to 10.
Judge only whether the solution does what the task asks, accurately.
"""
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
        output_format=Grade,
    ))
    return response.parsed_output


# Try it on one output before grading them all.
first = v1_outputs[0]
grade = grade_by_model(first["test_case"], first["output"])

print("\nScore     :", grade.score)
print("Strengths :", grade.strengths)
print("Weaknesses:", grade.weaknesses)
print("Reasoning :", grade.reasoning)

assert 1 <= grade.score <= 10

[input=1,261 output=248 cost=$0.00500 | spent $0.0353 of $0.30]

Score     : 9
Strengths : ['Correctly uses the list_objects_v2 paginator, so buckets with more than 1000 objects are fully listed', "Handles empty buckets safely with page.get('Contents', [])", 'Returns a plain list of key strings, exactly as the task specifies']
Weaknesses: ['Creates a new boto3 client on every call and offers no way to inject a client or session, which hurts testability', 'The optional error-handling variant wraps ClientError in a generic RuntimeError, which loses the original exception type for callers']
Reasoning : The main function is correct and idiomatic. It paginates properly, handles empty results, and returns the list of keys as requested. The weaknesses are minor design points that do not affect correctness for the stated task.


**What to notice**

- This is Claude grading Claude: one call writes the answer, and a second, separate call judges it.
- We put the task and solution in `<task>` and `<solution>` tags, so the grader can tell which text is which. Notebook 04 covers XML tags.
- Compare the `input` on this cost line with the earlier ones. The schema adds to every grading call, so **grading can cost as much as generating**.

## 4. Grader 2: a code grader (free)

A code grader is plain Python, so it's free, instant and always gives the same result. Each validator tries to parse the text: 10 if it works, 0 if it fails.

We can test these with no API call, so let's confirm they behave before trusting them.

In [7]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(test_case, output):
    validators = {"json": validate_json, "python": validate_python, "regex": validate_regex}
    return validators[test_case["format"]](output)


fenced_json = '```json\n{"a": 1}\n```'

checks = [
    ("Valid JSON", validate_json('{"a": 1}'), 10),
    ("JSON inside code fences", validate_json(fenced_json), 0),
    ("Valid Python", validate_python("def f():\n    return 1"), 10),
    ("Python with a sentence before it", validate_python("Here is the code:\ndef f(): pass"), 0),
    ("Valid regex", validate_regex(r"^arn:aws:iam::\d{12}:role/.+$"), 10),
    ("Broken regex", validate_regex("(unclosed"), 0),
]
for label, got, expected in checks:
    print(f"{'pass' if got == expected else 'FAIL'}  {label}: {got}")
    assert got == expected

print("\nA sentence checked as a regex scores:", validate_regex("Here is your regex, with an explanation."))

pass  Valid JSON: 10
pass  JSON inside code fences: 0
pass  Valid Python: 10
pass  Python with a sentence before it: 0
pass  Valid regex: 10
pass  Broken regex: 0

A sentence checked as a regex scores: 10


**What to notice**

- Code fences count as a failure, and that's on purpose. They make the output unusable by a program, so this one check covers both our **syntax** and **format** criteria.
- **Look at the last line.** Ordinary English is usually a valid regex, so `validate_regex` scores it 10. This grader is weak, and you'd only learn that by testing the grader itself. A real project would add stricter checks, and you can try that in Your turn.

> **Different from `claude-api-101`:** there I stripped code fences before validating, because Sonnet can't be prefilled. That hid the very failure we want to measure, so this version doesn't.

## 5. Putting it together: a baseline score (paid)

The final score is the average of the two grades, so both content and format count. You could weight them differently if one matters more to you.

`score_outputs` grades a list of outputs, and `print_results` shows them as a table. Here we grade the `v1_outputs` we already have, so only the model grader costs anything.

In [8]:
def score_outputs(outputs):
    results = []
    for item in outputs:
        test_case, output = item["test_case"], item["output"]
        grade = grade_by_model(test_case, output)
        syntax_score = grade_syntax(test_case, output)
        results.append({
            "test_case": test_case,
            "output": output,
            "model_score": grade.score,
            "syntax_score": syntax_score,
            "score": (grade.score + syntax_score) / 2,
            "reasoning": grade.reasoning,
        })
    return results


def print_results(results):
    print(f"\n{'score':>6}{'model':>7}{'syntax':>8}  task")
    for r in results:
        print(f"{r['score']:>6.1f}{r['model_score']:>7}{r['syntax_score']:>8}  [{r['test_case']['format']}] {r['test_case']['task'][:60]}")
    print(f"\nAverage score: {mean(r['score'] for r in results):.2f} / 10")


v1_results = score_outputs(v1_outputs)
print_results(v1_results)

assert len(v1_results) == len(dataset)

[input=1,261 output=301 cost=$0.00553 | spent $0.0409 of $0.30]
[input=1,047 output=302 cost=$0.00511 | spent $0.0460 of $0.30]
[input=1,224 output=338 cost=$0.00583 | spent $0.0518 of $0.30]
[input=1,271 output=259 cost=$0.00513 | spent $0.0569 of $0.30]

 score  model  syntax  task
   4.5      9       0  [python] Write a Python function `list_bucket_keys(bucket_name)` that
   4.5      9       0  [json] Create an IAM policy document as a JSON object that allows r
   4.5      9       0  [regex] Write a regex that matches a valid AWS S3 bucket name: 3 to 
   4.5      9       0  [python] Write a Python function `parse_arn(arn)` that takes an AWS A

Average score: 4.50 / 10


**What to notice**

- **When I ran this, every row scored 9 from the model grader and 0 for syntax.** All four answers were correct code, but each came wrapped in code fences and an explanation.
- **Look for rows with a high `model` score and a `syntax` of 0.** Those answers are correct, but a program can't use them. This is why you want both kinds of grader: the model grader doesn't see that problem, because we told it to judge only the content.
- The score has no meaning on its own. It's a **baseline**: what matters is whether a change moves it.

## 6. Version 2: clearer instructions (paid)

Now the payoff. We change only the prompt, and run the same exam.

In [9]:
def run_prompt_v2(test_case):
    prompt = f"""Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON, or a plain regex
* Do not add any comments, commentary, or explanation
* Do not use markdown code fences"""
    response = track(client.messages.create(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
    ))
    return text_from_message(response)


v2_results = score_outputs(get_outputs(dataset, run_prompt_v2))
print_results(v2_results)

[input=122 output=126 cost=$0.00150 | spent $0.0584 of $0.30]
[input=131 output=180 cost=$0.00206 | spent $0.0605 of $0.30]
[input=131 output=41 cost=$0.00067 | spent $0.0612 of $0.30]
[input=120 output=193 cost=$0.00217 | spent $0.0633 of $0.30]
[input=679 output=265 cost=$0.00401 | spent $0.0674 of $0.30]
[input=742 output=273 cost=$0.00421 | spent $0.0716 of $0.30]
[input=603 output=331 cost=$0.00452 | spent $0.0761 of $0.30]
[input=744 output=330 cost=$0.00479 | spent $0.0809 of $0.30]

 score  model  syntax  task
   9.5      9      10  [python] Write a Python function `list_bucket_keys(bucket_name)` that
  10.0     10      10  [json] Create an IAM policy document as a JSON object that allows r
   9.5      9      10  [regex] Write a regex that matches a valid AWS S3 bucket name: 3 to 
   9.5      9      10  [python] Write a Python function `parse_arn(arn)` that takes an AWS A

Average score: 9.62 / 10


## 7. Version 3: structured outputs (paid)

Version 2 relies on Claude following instructions. Notebook 02 showed a stronger option: put the answer in a field, so nothing can surround it.

> **Different from the course:** the course prefills `` ```code `` here, which current models reject.

In [10]:
class Solution(BaseModel):
    code: str


def run_prompt_v3(test_case):
    prompt = f"""Please solve the following task:

{test_case['task']}

Put only the solution in the code field: Python, JSON, or a plain regex, with no markdown fences."""
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
        output_format=Solution,
    ))
    return response.parsed_output.code


v3_results = score_outputs(get_outputs(dataset, run_prompt_v3))
print_results(v3_results)

[input=310 output=144 cost=$0.00206 | spent $0.0829 of $0.30]
[input=319 output=194 cost=$0.00258 | spent $0.0855 of $0.30]
[input=319 output=41 cost=$0.00105 | spent $0.0866 of $0.30]
[input=308 output=196 cost=$0.00258 | spent $0.0891 of $0.30]
[input=679 output=273 cost=$0.00409 | spent $0.0932 of $0.30]
[input=711 output=270 cost=$0.00412 | spent $0.0973 of $0.30]
[input=596 output=306 cost=$0.00425 | spent $0.1016 of $0.30]
[input=728 output=303 cost=$0.00449 | spent $0.1061 of $0.30]

 score  model  syntax  task
   9.5      9      10  [python] Write a Python function `list_bucket_keys(bucket_name)` that
  10.0     10      10  [json] Create an IAM policy document as a JSON object that allows r
   9.5      9      10  [regex] Write a regex that matches a valid AWS S3 bucket name: 3 to 
   9.5      9      10  [python] Write a Python function `parse_arn(arn)` that takes an AWS A

Average score: 9.62 / 10


## 8. Comparing the versions (free)

This table is what the whole pipeline is for.

In [11]:
versions = {
    "v1: plain": v1_results,
    "v2: clear instructions": v2_results,
    "v3: structured output": v3_results,
}

print(f"{'version':<26}{'score':>7}{'model':>7}{'syntax':>8}")
for name, results in versions.items():
    print(f"{name:<26}"
          f"{mean(r['score'] for r in results):>7.2f}"
          f"{mean(r['model_score'] for r in results):>7.2f}"
          f"{mean(r['syntax_score'] for r in results):>8.2f}")

print(f"\nThis notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

assert all(len(results) == len(dataset) for results in versions.values())

version                     score  model  syntax
v1: plain                    4.50   9.00    0.00
v2: clear instructions       9.62   9.25   10.00
v3: structured output        9.62   9.25   10.00

This notebook spent $0.1061 of its $0.30 budget.


**What to notice**

- **When I ran this, v1 scored 4.50, and v2 and v3 both scored 9.62.** The whole improvement came from the `syntax` column, which went from 0 to 10. Our changes targeted format, so that's where it should show up.
- **v2 and v3 tied, but v3 cost more.** Its schema added input tokens to every answer (about 310 instead of 122 in my run). Here, plain instructions were enough. Structured output earns its extra cost when a single badly formatted answer would break your program, because it's a guarantee, and instructions are only a request.
- The `model` column shows whether content quality held up. A change that fixes format but hurts accuracy wouldn't be a real improvement.

### How far to trust this

An eval gives you evidence, not proof. For this small one:

- **Four test cases is very few.** One odd answer moves the average a lot. Real evals use tens or hundreds of cases.
- **Model scores vary.** The same output can score differently between runs, so small differences may just be noise. Re-run section 6 and watch how much it moves.
- **Claude wrote the exam.** A generated dataset can miss the strange inputs real users send. Add real examples when you have them.
- **Graders can be wrong.** Section 4 showed one passing plain English as a regex. Test your graders too.

Even so, this is far better than trying a prompt once and hoping. You can now change a prompt and see in numbers whether it helped.

## Takeaways

1. **An eval is an exam:** a fixed dataset, graders, and a score you can compare.
2. **Save your dataset**, so every version is tested the same way and you pay for it only once.
3. **Decide what good means first**, then choose a grader for each criterion.
4. **Use both grader types.** Code graders are free and strict about form. Model graders can judge meaning.
5. **Ask for reasoning alongside the score**, or model graders tend to drift to the middle.
6. **Grading costs money too**, so reuse outputs when you can.
7. **Test your graders**, and don't over-read small differences on a small dataset.

## Your turn

Each exercise costs about 5 cents or less. The first two are free.

- **Free:** make `validate_regex` stricter, so it fails anything containing a space. Which of the checks in section 4 need updating?
- **Free:** in section 8, change the score to weight syntax twice as heavily as the model score, using the results you already have.
- Add a fourth version with a system prompt, and add it to the table in section 8.
- Delete `eval_dataset.json` and re-run from section 1. How much do the scores move with a different exam?

**Next:** [04 · Prompt engineering](04_prompt_engineering.ipynb): techniques for writing better prompts, each one measured with this kind of eval.